# Lab: Transformer Anatomy

In this lab, you'll inspect a real transformer model and verify everything from the chapter:
- Load a model config and examine the architecture
- Count parameters per component (attention vs MLP vs embedding)
- Verify weight matrix shapes match the chapter's derivations
- Compute memory breakdown and compare to the anchor values framework
- Visualize where the 16 GB lives

In [2]:
import sys
sys.path.insert(0, '../../..')
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoConfig

# We only need the config (no GPU required, no model download)
config = AutoConfig.from_pretrained('meta-llama/Llama-3.1-8B')

print('=== Llama 3.1 8B Config ===')
print(f'Hidden size (d):      {config.hidden_size}')
print(f'Layers (L):           {config.num_hidden_layers}')
print(f'Query heads:          {config.num_attention_heads}')
print(f'KV heads:             {config.num_key_value_heads}')
print(f'Head dim:             {config.hidden_size // config.num_attention_heads}')
print(f'Intermediate (MLP):   {config.intermediate_size}')
print(f'Vocab size:           {config.vocab_size}')
print(f'Max context:          {config.max_position_embeddings}')

/local/home/jharshul/work/llm-inference-at-scale/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.1-8B.
401 Client Error. (Request ID: Root=1-6a21b4c2-5bd9ca7d1ee8d0e2067ccc41;868c8b54-fccb-417d-9981-ed5a4dc4bfd3)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.1-8B/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B is restricted. You must have access to it and be authenticated to access it. Please log in.

## Verify: Deriving All Sizes from 3 Anchors

The chapter claims you only need `d`, `L`, and `P` to derive everything.
Let's verify using the actual config.

In [ ]:
# The three anchors
d = config.hidden_size          # 4096
L = config.num_hidden_layers     # 32
n_heads = config.num_attention_heads  # 32
n_kv_heads = config.num_key_value_heads  # 8
intermediate = config.intermediate_size  # 14336
vocab = config.vocab_size        # 128256

head_dim = d // n_heads
kv_dim = n_kv_heads * head_dim

print('=== Derived Values ===')
print(f'Head dim: {d} / {n_heads} = {head_dim}')
print(f'KV dim:   {n_kv_heads} × {head_dim} = {kv_dim}')
print(f'MLP ratio: {intermediate} / {d} = {intermediate/d:.1f}×')
print(f'GQA group size: {n_heads} / {n_kv_heads} = {n_heads // n_kv_heads} query heads per KV head')

## Attention Weight Shapes and Sizes

In [1]:
# Attention weights per layer
bytes_per_param = 2  # FP16

w_q = d * d * bytes_per_param
w_k = d * kv_dim * bytes_per_param
w_v = d * kv_dim * bytes_per_param
w_o = d * d * bytes_per_param
attn_per_layer = w_q + w_k + w_v + w_o

print('=== Attention Weights Per Layer ===')
print(f'W_q: [{d}, {d}]      = {w_q/1e6:.0f} MB')
print(f'W_k: [{d}, {kv_dim}]     = {w_k/1e6:.0f} MB')
print(f'W_v: [{d}, {kv_dim}]     = {w_v/1e6:.0f} MB')
print(f'W_o: [{d}, {d}]      = {w_o/1e6:.0f} MB')
print(f'─────────────────────────────────')
print(f'Total attention/layer: {attn_per_layer/1e6:.0f} MB')
print(f'Total attention (all {L} layers): {attn_per_layer * L / 1e9:.1f} GB')

NameError: name 'd' is not defined

## MLP Weight Shapes and Sizes

In [ ]:
# MLP weights per layer (SwiGLU = 3 matrices)
w_gate = d * intermediate * bytes_per_param
w_up = d * intermediate * bytes_per_param
w_down = intermediate * d * bytes_per_param
mlp_per_layer = w_gate + w_up + w_down

print('=== MLP Weights Per Layer (SwiGLU) ===')
print(f'W_gate: [{d}, {intermediate}] = {w_gate/1e6:.0f} MB')
print(f'W_up:   [{d}, {intermediate}] = {w_up/1e6:.0f} MB')
print(f'W_down: [{intermediate}, {d}] = {w_down/1e6:.0f} MB')
print(f'─────────────────────────────────')
print(f'Total MLP/layer: {mlp_per_layer/1e6:.0f} MB')
print(f'Total MLP (all {L} layers): {mlp_per_layer * L / 1e9:.1f} GB')

## Full Memory Breakdown

In [ ]:
# Complete model memory breakdown
embedding = vocab * d * bytes_per_param
lm_head = d * vocab * bytes_per_param
norms = L * 2 * d * bytes_per_param  # 2 RMSNorms per layer

total_attn = attn_per_layer * L
total_mlp = mlp_per_layer * L
total = total_attn + total_mlp + embedding + lm_head + norms

print('=== Where the Memory Lives ===')
print(f'Attention:       {total_attn/1e9:.1f} GB  ({100*total_attn/total:.0f}%)')
print(f'MLP:             {total_mlp/1e9:.1f} GB  ({100*total_mlp/total:.0f}%)')
print(f'Embedding:       {embedding/1e9:.2f} GB  ({100*embedding/total:.0f}%)')
print(f'LM Head:         {lm_head/1e9:.2f} GB  ({100*lm_head/total:.0f}%)')
print(f'RMSNorm:         {norms/1e6:.1f} MB  (<1%)')
print(f'─────────────────────────────────')
print(f'TOTAL:           {total/1e9:.1f} GB')
print(f'\nChapter claimed ~16 GB. Actual: {total/1e9:.1f} GB ✓' if abs(total/1e9 - 16) < 1 else f'\n⚠️ Mismatch! Got {total/1e9:.1f} GB')

In [ ]:
# Visualize the breakdown
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart: where parameters live
sizes = [total_attn, total_mlp, embedding + lm_head, norms]
labels = [f'Attention\n{total_attn/1e9:.1f} GB', f'MLP\n{total_mlp/1e9:.1f} GB',
          f'Embed+LM Head\n{(embedding+lm_head)/1e9:.1f} GB', 'Norms']
colors = ['#60a5fa', '#f97316', '#a78bfa', '#d1d5db']
explode = [0, 0.05, 0, 0]  # emphasize MLP
ax1.pie(sizes, labels=labels, colors=colors, explode=explode,
        autopct='%1.0f%%', startangle=90, textprops={'fontsize': 10})
ax1.set_title('Llama 3.1 8B: Parameter Distribution', fontsize=12)

# Bar chart: per-layer breakdown
components = ['W_q', 'W_k', 'W_v', 'W_o', 'W_gate', 'W_up', 'W_down']
sizes_mb = [w_q/1e6, w_k/1e6, w_v/1e6, w_o/1e6, w_gate/1e6, w_up/1e6, w_down/1e6]
colors_bar = ['#60a5fa']*4 + ['#f97316']*3
ax2.barh(components, sizes_mb, color=colors_bar)
ax2.set_xlabel('Size (MB per layer)')
ax2.set_title('Per-Layer Weight Sizes', fontsize=12)
ax2.axvline(x=sum(sizes_mb[:4])/4, color='#60a5fa', linestyle='--', alpha=0.5, label='Avg attention')
ax2.axvline(x=sum(sizes_mb[4:])/3, color='#f97316', linestyle='--', alpha=0.5, label='Avg MLP')
ax2.legend()

plt.tight_layout()
plt.savefig('images/memory_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: images/memory_breakdown.png')

## Exercise: Apply to a Different Model

Change the config to `meta-llama/Llama-3.1-70B` and re-run.
Verify that:
- Hidden size doubles (4096 → 8192)
- Layers increase (32 → 80)
- Total memory ≈ 140 GB
- The 70%/16%/13% ratio stays roughly the same

In [ ]:
# Try it: uncomment and run
# config_70b = AutoConfig.from_pretrained('meta-llama/Llama-3.1-70B')
# print(f'Hidden: {config_70b.hidden_size}, Layers: {config_70b.num_hidden_layers}')
# print(f'Expected memory: {config_70b.hidden_size * config_70b.num_hidden_layers * ...}')

## Key Takeaways

| What we verified | Result |
|-----------------|--------|
| Head dim = d / n_heads | 4096 / 32 = 128 ✓ |
| Attention per layer | 80 MB (Q:32 + K:8 + V:8 + O:32) ✓ |
| MLP per layer | 351 MB (gate:117 + up:117 + down:117) ✓ |
| MLP dominates | 70% of total parameters ✓ |
| Total model (FP16) | ~16 GB ✓ |
| GQA group size | 4 query heads per KV head ✓ |

**Next:** Module 0.1 — What happens when you actually *run* this architecture (prefill, decode, KV cache growth).